# 03 — Visualisation & Business Recommendations

**Project:** Gaming Launch Intelligence — Predicting Commercial Success and Optimising Platform & Launch Strategy

**Objective:** Create publication-quality interactive charts that directly answer the four core business questions, using both the Python-cleaned data and the SQL analytical outputs.

---

## Business Questions

1. **Which platform should we prioritise?**
2. **Which genre represents the strongest market opportunity?**
3. **Which regions should receive the largest marketing allocation?**
4. **What factors historically distinguish successful launches?**

---

## 1. Setup

In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

FIGURES_DIR = '../outputs/figures'
SQL_DIR = '../../SQL_Analysis/Result'

# Load Python-cleaned dataset
df = pd.read_csv('../outputs/cleaned_dataset.csv', parse_dates=['release_date', 'last_update'])
df['release_year'] = df['release_date'].dt.year.astype('Int64')

# Load SQL analytical outputs
genre_attract = pd.read_csv(f'{SQL_DIR}/genre_attractiveness.csv')
market_trend = pd.read_csv(f'{SQL_DIR}/market_trend.csv')
pub_conc = pd.read_csv(f'{SQL_DIR}/publisher_concentration.csv')
platform_fit = pd.read_csv(f'{SQL_DIR}/platform_fit.csv')
platform_life = pd.read_csv(f'{SQL_DIR}/platform_lifecycle.csv')
regional_opp = pd.read_csv(f'{SQL_DIR}/regional_opportunity.csv')
regional_corr = pd.read_csv(f'{SQL_DIR}/regional_correlation.csv')
genre_plat_fit = pd.read_csv(f'{SQL_DIR}/genre_platform_fit.csv')
launch_sim = pd.read_csv(f'{SQL_DIR}/launch_simulator.csv')
mktg_alloc = pd.read_csv(f'{SQL_DIR}/marketing_allocation.csv')

print(f'Cleaned dataset: {len(df):,} rows')
print(f'Launch scenarios: {len(launch_sim):,}')

Cleaned dataset: 18,850 rows
Launch scenarios: 92


---
## Q1: Which Platform Should We Prioritise?

### Platform Fit Scorecard

Each platform is scored on three dimensions: **efficiency** (sales per release), **presence** (catalog size), and **distribution** (how evenly sales are spread). The composite platform fit score weights efficiency at 50%, presence at 25%, and distribution at 25%.

In [12]:
# Radar chart for top platforms
top_platforms = platform_fit.nlargest(6, 'platform_fit_score')

categories = ['Efficiency', 'Presence', 'Distribution']

fig = go.Figure()
colors = px.colors.qualitative.Set2

for i, (_, row) in enumerate(top_platforms.iterrows()):
    values = [row['efficiency_score'], row['presence_score'], row['distribution_score']]
    values.append(values[0])  # close the polygon

    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill='toself',
        name=f"{row['platform']} ({row['platform_fit_score']:.0f})",
        line=dict(color=colors[i % len(colors)]),
        opacity=0.7
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Platform Fit Scorecard: Top 6 Platforms',
    height=550, width=650
)
fig.write_html(f'{FIGURES_DIR}/platform_radar.html')
fig.show()

In [13]:
# Platform lifecycle heatmap
pivot = platform_life.pivot_table(index='platform', columns='year', values='release_share_pct', fill_value=0)

# Filter to platforms with meaningful presence
platform_totals = pivot.sum(axis=1)
active_platforms = platform_totals[platform_totals > 10].index
pivot = pivot.loc[active_platforms]

# Sort by peak year
pivot = pivot.loc[pivot.idxmax(axis=1).sort_values().index]

fig = px.imshow(
    pivot, color_continuous_scale='YlGnBu',
    labels=dict(x='Year', y='Platform', color='Release Share (%)'),
    title='Platform Lifecycle Heatmap: Release Share Over Time',
    aspect='auto'
)
fig.update_layout(height=600)
fig.write_html(f'{FIGURES_DIR}/platform_lifecycle_heatmap.html')
fig.show()

In [14]:
# Platform sales distribution box plot
ANALYSIS_START, ANALYSIS_END = 2006, 2018
df_scoped = df[(df['release_year'] >= ANALYSIS_START) & (df['release_year'] <= ANALYSIS_END)]

top_plat = platform_fit.nlargest(8, 'platform_fit_score')['platform'].tolist()
plat_data = df_scoped[df_scoped['console'].isin(top_plat)]

fig = px.box(
    plat_data, x='console', y='total_sales', color='console',
    log_y=True, points=False,
    labels={'total_sales': 'Total Sales (M$, log)', 'console': 'Platform'},
    title='Sales Distribution by Top Platforms (Log Scale)',
    category_orders={'console': plat_data.groupby('console')['total_sales'].median().sort_values(ascending=False).index.tolist()}
)
fig.update_layout(height=450, showlegend=False)
fig.write_html(f'{FIGURES_DIR}/platform_box_comparison.html')
fig.show()

---
## Q2: Which Genre Represents the Strongest Market Opportunity?

### Genre Attractiveness Bubble Chart

Each genre is plotted with its sales productivity (y-axis), title-level concentration (x-axis), market trend (colour), and number of titles (bubble size). The ideal position is **top-left**: high sales, low concentration.

In [15]:
# Merge attractiveness with trend data
genre_combined = genre_attract.merge(market_trend[['genre', 'trend_score']], on='genre', how='left')
genre_combined['trend_score'] = genre_combined['trend_score'].fillna(50)

fig = px.scatter(
    genre_combined, x='concentration_norm', y='median_sales_per_title',
    size='title_count', color='trend_score',
    color_continuous_scale='RdYlGn',
    hover_name='genre',
    hover_data={'attractiveness_score': True, 'top10_share_pct': True},
    size_max=50,
    labels={
        'concentration_norm': 'Title Concentration (0=distributed, 1=dominated)',
        'median_sales_per_title': 'Median Sales per Title (M$)',
        'trend_score': 'Trend Score',
        'title_count': 'Title Count'
    },
    title='Genre Attractiveness: Sales Productivity vs. Concentration'
)
fig.update_layout(height=550)
fig.write_html(f'{FIGURES_DIR}/genre_attractiveness_bubble.html')
fig.show()

In [16]:
# Market share trend lines (faceted)
yearly_genre = (
    df_scoped[df_scoped['genre'].notna()]
    .groupby(['release_year', 'genre'])['total_sales']
    .sum()
    .reset_index()
)
yearly_total = df_scoped.groupby('release_year')['total_sales'].sum().reset_index()
yearly_total.columns = ['release_year', 'market_total']
yearly_genre = yearly_genre.merge(yearly_total, on='release_year')
yearly_genre['market_share_pct'] = yearly_genre['total_sales'] / yearly_genre['market_total'] * 100

# Top 8 genres
top_genres = genre_attract.nlargest(8, 'attractiveness_score')['genre'].tolist()

fig = px.line(
    yearly_genre[yearly_genre['genre'].isin(top_genres)],
    x='release_year', y='market_share_pct', color='genre',
    facet_col='genre', facet_col_wrap=4,
    labels={'release_year': 'Year', 'market_share_pct': 'Market Share (%)', 'genre': ''},
    title='Genre Market Share Trends (2006-2018)'
)
fig.update_layout(height=500, showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.write_html(f'{FIGURES_DIR}/genre_market_trends.html')
fig.show()

In [17]:
# Genre x Platform fit heatmap (from SQL results)
gp_pivot = genre_plat_fit.pivot_table(
    index='genre', columns='platform', values='median_sales_per_release', fill_value=0
)

fig = px.imshow(
    gp_pivot, color_continuous_scale='YlOrRd',
    labels=dict(x='Platform', y='Genre', color='Median Sales (M$)'),
    title='Genre x Platform Fit: Median Sales per Release',
    aspect='auto', text_auto='.2f'
)
fig.update_layout(height=500)
fig.write_html(f'{FIGURES_DIR}/genre_platform_heatmap.html')
fig.show()

---
## Q3: Which Regions Should Receive the Largest Marketing Allocation?

### Regional Demand Skew

Positive skew means a genre sells disproportionately well in that region compared to its global average — these are the regions where marketing spend will have the highest ROI.

In [18]:
# Regional skew chart
fig = px.bar(
    regional_opp, x='skew_pct', y='genre', color='region',
    orientation='h', barmode='group',
    color_discrete_map={'NA': '#2196F3', 'JP': '#FF5722', 'PAL': '#4CAF50', 'OTHER': '#9E9E9E'},
    labels={'skew_pct': 'Demand Skew (pp vs. global)', 'genre': '', 'region': 'Region'},
    title='Regional Demand Skew by Genre (Positive = Over-Index)'
)
fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.write_html(f'{FIGURES_DIR}/regional_skew.html')
fig.show()

In [19]:
# Regional correlation heatmap
corr_data = regional_corr.set_index('region')
corr_data.columns = ['NA', 'JP', 'PAL', 'Other', 'Global']

fig = px.imshow(
    corr_data.astype(float), color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1, text_auto='.2f',
    labels=dict(color='Correlation'),
    title='Regional Sales Correlation Matrix',
    aspect='auto'
)
fig.update_layout(height=450, width=550)
fig.write_html(f'{FIGURES_DIR}/regional_correlation_matrix.html')
fig.show()

In [20]:
# Marketing allocation treemap (for top genres)
top_alloc_genres = mktg_alloc.groupby('genre')['recommended_budget_pct'].sum().nlargest(8).index
alloc_filtered = mktg_alloc[
    mktg_alloc['genre'].isin(top_alloc_genres) &
    mktg_alloc['region'].notna() &
    (mktg_alloc['region'] != '') &
    mktg_alloc['recommended_budget_pct'].notna() &
    (mktg_alloc['recommended_budget_pct'] > 0)
].copy()
alloc_filtered['genre'] = alloc_filtered['genre'].astype(str).str.strip()
alloc_filtered['region'] = alloc_filtered['region'].astype(str).str.strip()

fig = px.treemap(
    alloc_filtered, path=[px.Constant('All Genres'), 'genre', 'region'], values='recommended_budget_pct',
    color='skew_pct', color_continuous_scale='RdYlGn',
    labels={'recommended_budget_pct': 'Budget Allocation (%)', 'skew_pct': 'Demand Skew (pp)'},
    title='Recommended Marketing Budget Allocation by Genre & Region'
)
fig.update_layout(height=550)
fig.write_html(f'{FIGURES_DIR}/marketing_allocation_treemap.html')
fig.show()

---
## Q4: What Factors Historically Distinguish Successful Launches?

### New-Entrant Performance

How do first-time publishers perform when entering a genre? The breakout rate measures the percentage of new entrants that achieve above-median sales in their debut year.

In [21]:
# Load new entrant data
entrant = pd.read_csv(f'{SQL_DIR}/new_entrant_performance.csv')

# Aggregated breakout rate by genre
entrant_agg = (
    entrant.groupby('genre')
    .agg(
        first_entrants=('publisher_name', 'count'),
        successful=('entry_performance', lambda x: x.isin(['strong breakout', 'above median']).sum())
    )
    .assign(breakout_rate=lambda x: (x['successful'] / x['first_entrants'] * 100).round(2))
    .sort_values('breakout_rate', ascending=False)
    .reset_index()
)

fig = px.bar(
    entrant_agg, x='breakout_rate', y='genre', orientation='h',
    color='breakout_rate', color_continuous_scale='RdYlGn',
    hover_data=['first_entrants', 'successful'],
    labels={'breakout_rate': 'Breakout Rate (%)', 'genre': '',
            'first_entrants': 'Total Entrants', 'successful': 'Successful'},
    title='New-Entrant Breakout Rate by Genre'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=450, showlegend=False)
fig.write_html(f'{FIGURES_DIR}/entrant_breakout_rate.html')
fig.show()

In [22]:
# Critic score impact on sales — violin plot by quartile
scored = df_scoped[df_scoped['critic_score'].notna()].copy()
scored['score_quartile'] = pd.qcut(scored['critic_score'], q=4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])

fig = px.violin(
    scored, x='score_quartile', y='total_sales', color='score_quartile',
    log_y=True, box=True, points=False,
    labels={'total_sales': 'Total Sales (M$, log)', 'score_quartile': 'Critic Score Quartile'},
    title='Sales Distribution by Critic Score Quartile'
)
fig.update_layout(height=450, showlegend=False)
fig.write_html(f'{FIGURES_DIR}/critic_score_violin.html')
fig.show()

---
## Launch Decision: Composite Scorecard

The launch simulator combines all six analytical dimensions into a weighted score for every genre-platform combination. Scenarios are classified as **GO** (>= 70), **CONDITIONAL** (55-69), or **AVOID** (< 55).

In [23]:
# Launch simulator horizontal stacked bar (top 15 scenarios)
top_scenarios = launch_sim.nlargest(15, 'launch_score')

dimensions = ['market_attractiveness_score', 'trend_score', 'publisher_opportunity_score',
              'entrant_accessibility_score', 'platform_fit_score', 'regional_opportunity_score']
dim_labels = ['Market Attractiveness (25%)', 'Trend (15%)', 'Publisher Opportunity (15%)',
              'Entrant Accessibility (20%)', 'Platform Fit (15%)', 'Regional Opportunity (10%)']
weights = [0.25, 0.15, 0.15, 0.20, 0.15, 0.10]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

top_scenarios['label'] = top_scenarios['genre'] + ' / ' + top_scenarios['platform']

fig = go.Figure()
for dim, label, weight, color in zip(dimensions, dim_labels, weights, colors):
    fig.add_trace(go.Bar(
        y=top_scenarios['label'],
        x=top_scenarios[dim] * weight,
        name=label,
        orientation='h',
        marker_color=color,
        text=top_scenarios[dim].round(0).astype(int),
        textposition='inside',
        textfont_size=9
    ))

fig.update_layout(
    barmode='stack',
    title='Top 15 Launch Scenarios — Weighted Score Breakdown',
    xaxis_title='Weighted Launch Score',
    yaxis={'categoryorder': 'total ascending'},
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, font_size=10)
)
fig.write_html(f'{FIGURES_DIR}/launch_scorecard.html')
fig.show()

In [24]:
# Decision summary table
decision_summary = (
    launch_sim.groupby('launch_decision')
    .agg(
        scenarios=('genre', 'count'),
        avg_score=('launch_score', 'mean'),
        best_genre=('genre', 'first'),
        best_platform=('platform', 'first')
    )
    .round(2)
    .reset_index()
)

print('Launch Decision Summary')
print('=' * 60)
for _, row in decision_summary.iterrows():
    print(f"  {row['launch_decision']:12s}  {row['scenarios']:3.0f} scenarios  avg score: {row['avg_score']:.1f}")

print(f"\nTotal scenarios evaluated: {len(launch_sim)}")
print(f"GO scenarios: {len(launch_sim[launch_sim['launch_decision'] == 'GO'])}")
print(f"CONDITIONAL:  {len(launch_sim[launch_sim['launch_decision'] == 'CONDITIONAL'])}")
print(f"AVOID:        {len(launch_sim[launch_sim['launch_decision'] == 'AVOID'])}")

Launch Decision Summary
  AVOID          62 scenarios  avg score: 48.4
  CONDITIONAL    30 scenarios  avg score: 60.0

Total scenarios evaluated: 92
GO scenarios: 0
CONDITIONAL:  30
AVOID:        62


In [25]:
# Top 10 recommended scenarios
top10 = launch_sim.nlargest(10, 'launch_score')[
    ['genre', 'platform', 'launch_score', 'launch_decision', 'recommended_region',
     'primary_risk', 'market_attractiveness_score', 'platform_fit_score']
].reset_index(drop=True)
top10.index = top10.index + 1
top10.columns = ['Genre', 'Platform', 'Score', 'Decision', 'Best Region',
                 'Primary Risk', 'Market Score', 'Platform Score']
top10

,Genre,Platform,Score,Decision,Best Region,Primary Risk,Market Score,Platform Score
1,Action-Adventure,X360,68.44,CONDITIONAL,PAL,No single dominant red flag,42.78,95.54
2,Action-Adventure,PS3,68.17,CONDITIONAL,PAL,No single dominant red flag,42.78,93.75
3,Action-Adventure,PS4,64.94,CONDITIONAL,PAL,No single dominant red flag,42.78,72.25
4,Strategy,X360,63.72,CONDITIONAL,JP,No single dominant red flag,40.24,95.54
5,Strategy,PS3,63.45,CONDITIONAL,JP,No single dominant red flag,40.24,93.75
6,Shooter,X360,62.58,CONDITIONAL,PAL,High entry risk,43.89,95.54
7,Action-Adventure,XONE,62.44,CONDITIONAL,PAL,No single dominant red flag,42.78,55.56
8,Shooter,PS3,62.31,CONDITIONAL,PAL,High entry risk,43.89,93.75
9,Role-Playing,X360,61.87,CONDITIONAL,JP,No single dominant red flag,41.21,95.54
10,Role-Playing,PS3,61.61,CONDITIONAL,JP,No single dominant red flag,41.21,93.75


---
## Executive Recommendations

Based on the quantitative analysis across 18,900+ game releases (2006-2018):

### 1. Platform Strategy
The platform fit analysis and statistical tests reveal significant sales differences between platforms. Platforms with high efficiency scores (high sales per release) and broad sales distribution should be prioritised. Declining platforms (visible in the lifecycle heatmap) should be avoided for new launches.

### 2. Genre Selection
Genres with high attractiveness scores combine strong per-title sales with distributed (non-concentrated) revenue. The trend analysis shows which genres have growing market share momentum. The ideal launch genre has:
- Above-average median sales per title
- Low title-level HHI (not dominated by a few blockbusters)
- Positive market share trend
- High new-entrant breakout rate (historical precedent for success)

### 3. Regional Marketing Allocation
Regional demand skew should drive marketing budget allocation. Genres that over-index in specific regions (positive skew) will deliver higher ROI on marketing spend in those regions. The correlation matrix shows that NA and PAL markets are highly correlated, while JP behaves independently — suggesting that a JP-focused strategy requires distinct content and positioning.

### 4. Success Factors
Statistical analysis reveals that publisher track record is the strongest predictor of commercial success, followed by platform popularity and critic score. For a new entrant:
- **Invest in quality:** Higher critic scores correlate with significantly higher sales
- **Choose accessible genres:** Genres with high breakout rates give new publishers the best odds
- **Target the right platform-genre combination:** The genre x platform fit heatmap identifies the highest-ceiling combinations

### Key Risk Flags
The launch simulator flags the single weakest dimension for each scenario as `primary_risk`. Common risk patterns include:
- **High entry risk:** The genre historically punishes new entrants
- **High publisher concentration:** A few publishers dominate the genre
- **Weak platform fit:** The platform doesn't generate strong per-title revenue
- **Declining trend:** The genre is losing market share

---
*Analysis conducted using VGChartz 2024 data. All sales figures in millions USD.*